### Preparation Notebook - Exposure

This notebook prepares the exposure datasets by disaggregating sectoral capital stock and GVA data

In [1]:
# Import live code changes in
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import geopandas as gpd
import rasterio
import pandas as pd
from functools import reduce

from sovereign.utils import disaggregate_total_to_raster, disaggregate_admin_to_raster

##### 1. Set filepaths and variables

In [2]:
root = Path.cwd().parent.parent # find project root
# Inputs
AGR_path = os.path.join(root, 'inputs', 'exposure', 'gridded', 'THA_cropland.tif')
INF_path = os.path.join(root, 'inputs', 'exposure', 'gridded', 'THA_infra.tif')
RES_path = os.path.join(root, 'inputs', 'exposure', 'gridded', 'THA_ghs-res_v.tif')
NRES_path = os.path.join(root, 'inputs', 'exposure', 'gridded', 'THA_ghs-nres_v.tif')
capstock_path = os.path.join(root, 'inputs', 'exposure', 'capstock', 'Thailand_2020_net_capital_stock_million_baht.csv')
gva_path = os.path.join(root, 'inputs', 'exposure', 'gva', 'DOSE_V2p11_THA_rgva.gpkg')
# Outputs
agr_pri_path = os.path.join(root, 'outputs', 'exposure', 'agr_priv_capstock.tif')
agr_pub_path = os.path.join(root, 'outputs', 'exposure', 'agr_pub_capstock.tif')
inf_pub_path = os.path.join(root, 'outputs', 'exposure', 'inf_pub_capstock.tif')
nres_pri_path = os.path.join(root, 'outputs', 'exposure', 'nres_priv_capstock.tif')
res_pri_path = os.path.join(root, 'outputs', 'exposure', 'res_priv_capstock.tif')
agr_gva_path = os.path.join(root, 'outputs', 'exposure', 'agr_GVA.tif')
man_gva_path = os.path.join(root, 'outputs', 'exposure', 'man_GVA.tif')
ser_gva_path = os.path.join(root, 'outputs', 'exposure', 'ser_GVA.tif')
# Important variables
exchange_rate = 0.032 # average Baht > USD exchange rate for 2020
residential_split = 0.28 # 28% residential / non-residential split taken from GIRI capital stock tables

##### 2. Disaggregate national capital stock over grid

In [3]:
# Load capstock data
capstock = pd.read_csv(capstock_path)
# Private Agriculture
agr_pri = capstock.loc[capstock['Sector']=='Agriculture', "Private"].iloc[0] * 1e6 * exchange_rate
disaggregate_total_to_raster(agr_pri, AGR_path, agr_pri_path)
# Public Agriculture
agr_pub = capstock.loc[capstock['Sector']=='Agriculture', "Public"].iloc[0] * 1e6 * exchange_rate
disaggregate_total_to_raster(agr_pub, AGR_path, agr_pub_path)
# Public Infrastructure
inf_pub = (capstock.loc[capstock['Sector']=='Industrial', "Public"].iloc[0] 
           + capstock.loc[capstock['Sector']=='Service', "Public"].iloc[0]
          ) * 1e6 * exchange_rate
disaggregate_total_to_raster(inf_pub, INF_path, inf_pub_path)
# Private Buildings
private_buildings = (capstock.loc[capstock['Sector']=='Industrial', "Private"].iloc[0] 
           + capstock.loc[capstock['Sector']=='Service', "Private"].iloc[0]
          ) * 1e6 * exchange_rate
# Residential
res_pri = private_buildings * residential_split
disaggregate_total_to_raster(res_pri, RES_path, res_pri_path)
# Non-Residential
nres_pri = private_buildings * (1-residential_split)
disaggregate_total_to_raster(nres_pri, NRES_path, nres_pri_path)

##### 3. Disaggregate sub-national GVA over grid

In [4]:
# Agriculture
disaggregate_admin_to_raster(gva_path, 'ag_grp_usd', AGR_path, agr_gva_path)
# Manufacturing
disaggregate_admin_to_raster(gva_path, 'man_grp_usd', NRES_path, man_gva_path)
# Service
disaggregate_admin_to_raster(gva_path, 'serv_grp_usd', NRES_path, ser_gva_path)